## Model accuracy
Import libraries

In [1]:
import os
import json
import utils
import numpy as np
import pandas as pd
import tensorflow as tf

from silx.io.dictdump import h5todict
from scipy.constants import Avogadro as UNA

Define inputs

In [2]:
model_name = "meno"
# Paths
path_to_datagen = "./../00.datagen/"
paths = {
  "data": path_to_datagen + "/data/test/",
  "model": "./../03.merged/case.r01.noref/saving/noneq/SavedModel/",
  "saving": "./output/error/"
}
os.makedirs(paths["saving"], exist_ok=True)
# Error
eps = 1e-10

In [3]:
# Data indices
with open(path_to_datagen + "/data/indices.json") as file:
  _indices = json.load(file)
indices = {
  "x": [
    _indices["na_0"],
    _indices["nmi_0"],
    _indices["T_0"],
    _indices["t"],
    _indices["qoi_eq"]
  ],
  "y": [
    _indices["na"],
    _indices["nmi"],
    _indices["T"]
  ]
}

In [4]:
# Time grid
tlim = [1e-11, 1e-1]
tdim = 200
t = np.geomspace(*tlim, num=tdim)

Load data and model

In [ ]:
data = utils.load_test_cases(paths["data"])
model = tf.keras.models.load_model(paths["model"])

Set variables names

In [6]:
names = np.array(list(data.values())[0].columns)
names = np.concatenate([names[i] for i in indices["y"]]).tolist()

In [7]:
names_latex = {}
for name in names:
  if ("O2" in name):
    names_latex[name] = "O$_2\,(%s)$" % name.split("_")[-1]
  elif ("O" in name):
    names_latex[name] = name.split("_")[-1]
  else:
    names_latex[name] = "$%s$" % name

Compute error

In [8]:
scale = np.full(len(names), 1.0/UNA)
scale[-1] = 1.0
scale = scale.reshape(1,-1)

In [9]:
error = {"l2": [], "mape": []}
for idata in data.values():
  _, y_true, y_pred = utils.predict(model, idata.to_numpy(), indices, t)
  y_true *= scale
  y_pred *= scale
  error["l2"].append(utils.error_l2(y_true, y_pred, eps=eps))
  error["mape"].append(utils.error_mape(y_true, y_pred, eps=eps))

In [10]:
for (k, err) in error.items():
  kerr = np.vstack(err)
  kerr = np.vstack([np.mean(kerr, axis=0), np.std(kerr, axis=0)])
  kerr = pd.DataFrame(kerr, columns=names, index=["mean", "std"])
  kerr = kerr.sort_values(by=['mean'], axis=1)
  kerr.to_csv(paths["saving"]+f"/{model_name}_{k}.csv", float_format="%.3f")
  utils.write_latex(
    error=kerr,
    filename=paths["saving"]+f"/{model_name}_{k}_latex.txt",
    names=names_latex
  )